In [4]:
%pip install pandas numpy scikit-learn folium joblib pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 5.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 6.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 5.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 6.6 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import folium
import joblib
import pyarrow.dataset as ds

from folium.plugins import TimestampedGeoJson
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
INPUT_PARQUET = "street.parquet"

DATE_COL = "Month"
LSOA_COL = "LSOA code"
LAT_COL = "Latitude"
LON_COL = "Longitude"
CRIME_TYPE_COL = "Crime type"
FORCE_COL = "Falls within"

TARGET_CRIME_TYPES = ["Vehicle crime", "Other theft", "Robbery", "Public order", "Other crime", "Burglary", "Violence and sexual offences"]

CATEGORY = "Police officers"

HOTSPOT_PERCENTILE = 0.70
MIN_HOTSPOT_DEMAND = 1

OUTPUT_AGGREGATED_PARQUET = "police_officers.parquet"
OUTPUT_MODEL = "police_officers.joblib"
OUTPUT_FORECAST_CSV = "police_officers.csv"
OUTPUT_HOTSPOT_CSV = "police_officers.csv"
OUTPUT_MAP_HTML = "police_officers_hotspots.html"

In [3]:
dataset = ds.dataset(INPUT_PARQUET, format="parquet")

available_columns = dataset.schema.names
print("Available columns:")
print(available_columns)

columns_to_load = [
    DATE_COL,
    LSOA_COL,
    LAT_COL,
    LON_COL,
    CRIME_TYPE_COL
]

if FORCE_COL in available_columns:
    columns_to_load.append(FORCE_COL)

print("\nLoading these crime types:")
for crime in TARGET_CRIME_TYPES:
    print("-", crime)

table = dataset.to_table(
    columns=columns_to_load,
    filter=ds.field(CRIME_TYPE_COL).isin(TARGET_CRIME_TYPES)
)

combined_raw = table.to_pandas()

combined_raw["combined_crime_type"] = CATEGORY

print("\nLoaded rows:", combined_raw.shape)
print("\nCrime type counts:")
print(combined_raw[CRIME_TYPE_COL].value_counts())

combined_raw.head()

Available columns:
['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']

Loading these crime types:
- Vehicle crime
- Other theft
- Robbery
- Public order
- Other crime
- Burglary
- Violence and sexual offences

Loaded rows: (49048175, 7)

Crime type counts:
Crime type
Violence and sexual offences    21501492
Other theft                      7295446
Vehicle crime                    5809744
Burglary                         5501279
Public order                     4767827
Other crime                      3131274
Robbery                          1041113
Name: count, dtype: int64


,Month,LSOA code,Latitude,Longitude,Crime type,Falls within,combined_crime_type
0,2010-12,E01017662,51.819143,-0.805600,Other crime,Avon and Somerset Constabulary,Police officers
1,2010-12,E01014399,51.409456,-2.513308,Burglary,Avon and Somerset Constabulary,Police officers
2,2010-12,E01014399,51.416137,-2.509126,Burglary,Avon and Somerset Constabulary,Police officers
3,2010-12,E01014399,51.418169,-2.494366,Other crime,Avon and Somerset Constabulary,Police officers
4,2010-12,E01014400,51.415233,-2.497799,Burglary,Avon and Somerset Constabulary,Police officers


In [4]:
combined_raw[DATE_COL] = pd.to_datetime(
    combined_raw[DATE_COL],
    format="%Y-%m",
    errors="coerce"
)

combined_raw["month"] = (
    combined_raw[DATE_COL]
    .dt.to_period("M")
    .dt.to_timestamp()
)

group_cols = ["month", LSOA_COL]

if FORCE_COL in combined_raw.columns:
    group_cols.append(FORCE_COL)

monthly = (
    combined_raw.groupby(group_cols, as_index=False)
    .agg(
        demand=(LSOA_COL, "size"),
        Latitude=(LAT_COL, "median"),
        Longitude=(LON_COL, "median")
    )
)

monthly["combined_crime_type"] = CATEGORY

monthly.to_parquet(OUTPUT_AGGREGATED_PARQUET, index=False)

print("Monthly combined demand created.")
print("Rows:", monthly.shape)
print("Combined crime name:", CATEGORY)
print("\nMonthly demand summary:")
print(monthly["demand"].describe())

monthly.head()

Monthly combined demand created.
Rows: (6172994, 7)
Combined crime name: Police officers

Monthly demand summary:
count    6.172994e+06
mean     7.642995e+00
std      1.011806e+01
min      1.000000e+00
25%      3.000000e+00
50%      5.000000e+00
75%      9.000000e+00
max      8.190000e+02
Name: demand, dtype: float64


,month,LSOA code,Falls within,demand,Latitude,Longitude,combined_crime_type
0,2010-12-01,E01000001,City of London Police,2,51.517332,-0.098519,Police officers
1,2010-12-01,E01000002,City of London Police,14,51.516922,-0.093948,Police officers
2,2010-12-01,E01000003,City of London Police,4,51.521335,-0.095808,Police officers
3,2010-12-01,E01000005,City of London Police,21,51.512650,-0.075613,Police officers
4,2010-12-01,E01000006,Metropolitan Police Service,5,51.540005,0.088021,Police officers


In [5]:
lsoa_meta = (
    monthly[[LSOA_COL, LAT_COL, LON_COL]]
    .groupby(LSOA_COL, as_index=False)
    .agg(
        Latitude=(LAT_COL, "median"),
        Longitude=(LON_COL, "median")
    )
)

print(lsoa_meta.shape)
lsoa_meta.head()

(36745, 3)


,LSOA code,Latitude,Longitude
0,E01000001,51.519138,-0.097467
1,E01000002,51.518743,-0.091498
2,E01000003,51.521660,-0.095648
3,E01000005,51.513061,-0.075401
4,E01000006,51.538745,0.087964


In [6]:
monthly = (
    monthly.groupby(["month", LSOA_COL], as_index=False)
           .agg(
               demand=("demand", "sum"),
               Latitude=(LAT_COL, "median"),
               Longitude=(LON_COL, "median")
           )
)

all_months = pd.date_range(
    monthly["month"].min(),
    monthly["month"].max(),
    freq="MS"
)

all_lsoas = monthly[LSOA_COL].unique()

panel_index = pd.MultiIndex.from_product(
    [all_months, all_lsoas],
    names=["month", LSOA_COL]
)

panel = (
    monthly.set_index(["month", LSOA_COL])
           .reindex(panel_index)
           .reset_index()
)

panel["demand"] = panel["demand"].fillna(0)

panel = panel.drop(columns=[LAT_COL, LON_COL], errors="ignore")
panel = panel.merge(lsoa_meta, on=LSOA_COL, how="left")

print(panel.shape)
panel.head()

(6761080, 5)


,month,LSOA code,demand,Latitude,Longitude
0,2010-12-01,E01000001,2.0,51.519138,-0.097467
1,2010-12-01,E01000002,14.0,51.518743,-0.091498
2,2010-12-01,E01000003,4.0,51.521660,-0.095648
3,2010-12-01,E01000005,21.0,51.513061,-0.075401
4,2010-12-01,E01000006,5.0,51.538745,0.087964


In [7]:
def add_features(data):
    data = data.sort_values([LSOA_COL, "month"]).copy()

    data["year"] = data["month"].dt.year
    data["month_num"] = data["month"].dt.month
    data["quarter"] = data["month"].dt.quarter

    data["month_sin"] = np.sin(2 * np.pi * data["month_num"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month_num"] / 12)

    for lag in [1, 2, 3, 6, 12]:
        data[f"lag_{lag}"] = data.groupby(LSOA_COL)["demand"].shift(lag)

    data["rolling_3_mean"] = (
        data.groupby(LSOA_COL)["demand"]
        .transform(lambda x: x.shift(1).rolling(3).mean())
    )

    data["rolling_6_mean"] = (
        data.groupby(LSOA_COL)["demand"]
        .transform(lambda x: x.shift(1).rolling(6).mean())
    )

    data["rolling_12_mean"] = (
        data.groupby(LSOA_COL)["demand"]
        .transform(lambda x: x.shift(1).rolling(12).mean())
    )

    data["rolling_3_std"] = (
        data.groupby(LSOA_COL)["demand"]
        .transform(lambda x: x.shift(1).rolling(3).std())
    )

    data["rolling_12_std"] = (
        data.groupby(LSOA_COL)["demand"]
        .transform(lambda x: x.shift(1).rolling(12).std())
    )

    data["lsoa_mean_demand"] = (
        data.groupby(LSOA_COL)["demand"]
        .transform(lambda x: x.shift(1).expanding().mean())
    )

    fill_cols = [
        "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
        "rolling_3_mean", "rolling_6_mean", "rolling_12_mean",
        "rolling_3_std", "rolling_12_std",
        "lsoa_mean_demand"
    ]

    data[fill_cols] = data[fill_cols].fillna(0)

    return data

In [8]:
model_data = add_features(panel)

feature_cols = [
    "year",
    "month_num",
    "quarter",
    "month_sin",
    "month_cos",
    LAT_COL,
    LON_COL,
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
    "rolling_3_mean",
    "rolling_6_mean",
    "rolling_12_mean",
    "rolling_3_std",
    "rolling_12_std",
    "lsoa_mean_demand"
]

print(model_data.shape)
model_data.head()

(6761080, 21)


,month,LSOA code,demand,Latitude,Longitude,year,month_num,quarter,month_sin,month_cos,...,lag_2,lag_3,lag_6,lag_12,rolling_3_mean,rolling_6_mean,rolling_12_mean,rolling_3_std,rolling_12_std,lsoa_mean_demand
0,2010-12-01,E01000001,2.0,51.519138,-0.097467,2010,12,4,-2.449294e-16,1.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.00
36745,2011-01-01,E01000001,7.0,51.519138,-0.097467,2011,1,1,5.000000e-01,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,2.00
73490,2011-02-01,E01000001,6.0,51.519138,-0.097467,2011,2,1,8.660254e-01,5.000000e-01,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,4.50
110235,2011-03-01,E01000001,8.0,51.519138,-0.097467,2011,3,1,1.000000e+00,6.123234e-17,...,7.0,2.0,0.0,0.0,5.0,0.0,0.0,2.645751,0.0,5.00
146980,2011-04-01,E01000001,5.0,51.519138,-0.097467,2011,4,2,8.660254e-01,-5.000000e-01,...,6.0,7.0,0.0,0.0,7.0,0.0,0.0,1.000000,0.0,5.75


In [9]:
train = model_data[
    model_data["month"] < "2025-01-01"
]

test = model_data[
    (model_data["month"] >= "2025-01-01") &
    (model_data["month"] <= "2026-03-01")
]

X_train = train[feature_cols]
y_train = train["demand"]

X_test = test[feature_cols]
y_test = test["demand"]

print("Train rows:", len(train))
print("Test rows:", len(test))

Train rows: 6209905
Test rows: 551175


In [10]:
model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

model.fit(X_train, y_train)

joblib.dump(model, OUTPUT_MODEL)

print("Model trained and saved:", OUTPUT_MODEL)

Model trained and saved: police_officers.joblib


In [11]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

test_preds = model.predict(X_test)
test_preds = np.maximum(test_preds, 0)

mae = mean_absolute_error(y_test, test_preds)
rmse = mean_squared_error(y_test, test_preds) ** 0.5
r2 = r2_score(y_test, test_preds)

print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R²:", round(r2, 3))

MAE: 2.58
RMSE: 3.847
R²: 0.866


In [12]:
history = panel.copy()

forecast_months = pd.date_range(
    "2027-01-01",
    "2027-12-01",
    freq="MS"
)

forecast_results = []

for forecast_month in forecast_months:
    print("Forecasting:", forecast_month.strftime("%Y-%m"))

    future_rows = pd.DataFrame({
        "month": forecast_month,
        LSOA_COL: history[LSOA_COL].unique()
    })

    future_rows = future_rows.merge(lsoa_meta, on=LSOA_COL, how="left")
    future_rows["demand"] = np.nan

    temp = pd.concat([history, future_rows], ignore_index=True)
    temp = add_features(temp)

    predict_rows = temp[temp["month"] == forecast_month].copy()

    X_future = predict_rows[feature_cols]

    predicted = model.predict(X_future)
    predicted = np.maximum(predicted, 0)

    predict_rows["predicted_demand"] = predicted.round(2)

    forecast_results.append(
        predict_rows[
            ["month", LSOA_COL, LAT_COL, LON_COL, "predicted_demand"]
        ]
    )

    new_history_rows = predict_rows[
        ["month", LSOA_COL, LAT_COL, LON_COL]
    ].copy()

    new_history_rows["demand"] = predict_rows["predicted_demand"].values

    history = pd.concat([history, new_history_rows], ignore_index=True)

forecast_df = pd.concat(forecast_results, ignore_index=True)

forecast_df.head()

Forecasting: 2027-01
Forecasting: 2027-02
Forecasting: 2027-03
Forecasting: 2027-04
Forecasting: 2027-05
Forecasting: 2027-06
Forecasting: 2027-07
Forecasting: 2027-08
Forecasting: 2027-09
Forecasting: 2027-10
Forecasting: 2027-11
Forecasting: 2027-12


,month,LSOA code,Latitude,Longitude,predicted_demand
0,2027-01-01,E01000001,51.519138,-0.097467,9.25
1,2027-01-01,E01000002,51.518743,-0.091498,17.96
2,2027-01-01,E01000003,51.521660,-0.095648,7.36
3,2027-01-01,E01000005,51.513061,-0.075401,39.39
4,2027-01-01,E01000006,51.538745,0.087964,6.78


In [13]:
forecast_df["hotspot_threshold"] = (
    forecast_df.groupby("month")["predicted_demand"]
               .transform(lambda x: x.quantile(HOTSPOT_PERCENTILE))
)

forecast_df["is_hotspot"] = (
    (forecast_df["predicted_demand"] >= forecast_df["hotspot_threshold"]) &
    (forecast_df["predicted_demand"] >= MIN_HOTSPOT_DEMAND)
)

hotspots_df = forecast_df[forecast_df["is_hotspot"]].copy()

print("All forecast rows:", len(forecast_df))
print("Hotspot rows:", len(hotspots_df))

hotspots_df.head()

All forecast rows: 440940
Hotspot rows: 132418


,month,LSOA code,Latitude,Longitude,predicted_demand,hotspot_threshold,is_hotspot
0,2027-01-01,E01000001,51.519138,-0.097467,9.25,8.14,True
1,2027-01-01,E01000002,51.518743,-0.091498,17.96,8.14,True
3,2027-01-01,E01000005,51.513061,-0.075401,39.39,8.14,True
5,2027-01-01,E01000007,51.539500,0.080477,29.30,8.14,True
6,2027-01-01,E01000008,51.540025,0.071568,20.03,8.14,True


In [14]:
forecast_df.to_csv(OUTPUT_FORECAST_CSV, index=False)
hotspots_df.to_csv(OUTPUT_HOTSPOT_CSV, index=False)

print("Saved:")
print(OUTPUT_FORECAST_CSV)
print(OUTPUT_HOTSPOT_CSV)

Saved:
police_officers.csv
police_officers.csv


In [16]:
m = folium.Map(
    location=[54.5, -3.0],
    zoom_start=5,
    tiles="cartodbpositron"
)

features = []

for _, row in hotspots_df.iterrows():

    popup_text = f"""
    <b>Crime types:</b> Anti-social behaviour, Shoplifting, and Public order offences<br>
    <b>LSOA:</b> {row[LSOA_COL]}<br>
    <b>Predicted demand:</b> {row['predicted_demand']}<br>
    <b>Month:</b> {row['month'].strftime('%Y-%m')}<br>
    <b>Hotspot definition:</b> Top 10% highest demand
    """

    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [
                row[LON_COL],
                row[LAT_COL]
            ],
        },
        "properties": {
            "times": [
                row["month"].strftime("%Y-%m-%d")
            ],
            "icon": "circle",
            "iconstyle": {
                "fillColor": "red",
                "fillOpacity": 0.65,
                "stroke": True,
                "color": "darkred",
                "weight": 1,
                "radius": 5
            },
            "popup": popup_text
        }
    }

    features.append(feature)

TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": features,
    },
    period="P1M",
    duration="P1M",
    add_last_point=False,
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM",
    time_slider_drag_update=True,
).add_to(m)

In [17]:
m.save(OUTPUT_MAP_HTML)

print("Saved map:")
print(OUTPUT_MAP_HTML)

Saved map:
police_officers_hotspots.html


This is the allocation part of the model.

In [31]:
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd

LAT_COL = "Latitude"
LON_COL = "Longitude"
DEMAND_COL = "predicted_demand"

BIKE_SPEED_KMH = 15
CAR_SPEED_KMH = 60

# Dense hotspot definition
# If a hotspot has at least 5 other hotspots within 1 km, it is considered dense
DENSITY_EPS_KM = 1.0
DENSITY_MIN_SAMPLES = 5

OUTPUT_PATROL_ALLOCATIONS = "police_officers_allocation.csv"
OUTPUT_ALLOCATION_MAP_HTML = "police_officers_allocation.html"

CENTROIDS = 8156

print("Settings loaded.")
print("Bike speed:", BIKE_SPEED_KMH, "km/h")
print("Car speed:", CAR_SPEED_KMH, "km/h")

Settings loaded.
Bike speed: 15 km/h
Car speed: 60 km/h


In [32]:
def latlon_to_xy_km(lat, lon, ref_lat):
    """
    Converts latitude/longitude into approximate x/y coordinates in kilometres.
    This is good enough for clustering within UK-scale areas.
    """
    x = lon * 111.320 * np.cos(np.radians(ref_lat))
    y = lat * 110.574
    return np.column_stack([x, y])


def xy_km_to_latlon(x, y, ref_lat):
    """
    Converts x/y kilometre coordinates back to latitude/longitude.
    """
    lat = y / 110.574
    lon = x / (111.320 * np.cos(np.radians(ref_lat)))
    return lat, lon


def speed_to_minutes_per_km(speed_kmh):
    """
    Converts speed in km/h to minutes needed per kilometre.
    """
    return 60 / speed_kmh

In [33]:
classified_hotspots = []

for month, month_df in hotspots_df.groupby("month"):
    month_df = month_df.copy()
    month_name = pd.to_datetime(month).strftime("%Y-%m")

    print("\nClassifying patrol type for:", month_name)
    print("Hotspots in month:", len(month_df))

    ref_lat = month_df[LAT_COL].mean()

    coords_km = latlon_to_xy_km(
        month_df[LAT_COL].values,
        month_df[LON_COL].values,
        ref_lat
    )

    density_model = DBSCAN(
        eps=DENSITY_EPS_KM,
        min_samples=DENSITY_MIN_SAMPLES
    )

    density_labels = density_model.fit_predict(coords_km)

    month_df["density_cluster"] = density_labels

    month_df["patrol_type"] = np.where(
        month_df["density_cluster"] == -1,
        "Car patrol",
        "Bike patrol"
    )

    print(month_df["patrol_type"].value_counts())

    classified_hotspots.append(month_df)

classified_hotspots_df = pd.concat(classified_hotspots, ignore_index=True)

print("\nClassification complete.")
print("Total classified hotspots:", len(classified_hotspots_df))
classified_hotspots_df.head()


Classifying patrol type for: 2027-01
Hotspots in month: 11027
patrol_type
Bike patrol    7311
Car patrol     3716
Name: count, dtype: int64

Classifying patrol type for: 2027-02
Hotspots in month: 11037
patrol_type
Bike patrol    7334
Car patrol     3703
Name: count, dtype: int64

Classifying patrol type for: 2027-03
Hotspots in month: 11049
patrol_type
Bike patrol    7320
Car patrol     3729
Name: count, dtype: int64

Classifying patrol type for: 2027-04
Hotspots in month: 11048
patrol_type
Bike patrol    7348
Car patrol     3700
Name: count, dtype: int64

Classifying patrol type for: 2027-05
Hotspots in month: 11030
patrol_type
Bike patrol    7364
Car patrol     3666
Name: count, dtype: int64

Classifying patrol type for: 2027-06
Hotspots in month: 11027
patrol_type
Bike patrol    7403
Car patrol     3624
Name: count, dtype: int64

Classifying patrol type for: 2027-07
Hotspots in month: 11026
patrol_type
Bike patrol    7402
Car patrol     3624
Name: count, dtype: int64

Classifying 

,month,LSOA code,Latitude,Longitude,predicted_demand,hotspot_threshold,is_hotspot,density_cluster,patrol_type
0,2027-01-01,E01000001,51.519138,-0.097467,9.25,8.14,True,0,Bike patrol
1,2027-01-01,E01000002,51.518743,-0.091498,17.96,8.14,True,0,Bike patrol
2,2027-01-01,E01000005,51.513061,-0.075401,39.39,8.14,True,0,Bike patrol
3,2027-01-01,E01000007,51.539500,0.080477,29.30,8.14,True,0,Bike patrol
4,2027-01-01,E01000008,51.540025,0.071568,20.03,8.14,True,0,Bike patrol


In [34]:
from scipy.spatial import cKDTree
import numpy as np
import pandas as pd

def initialize_weighted_medoids(coords, weights, k, random_state=42):
    """
    Selects initial medoids from actual data points.
    Points with higher predicted demand have a higher chance to be selected.
    """
    rng = np.random.default_rng(random_state)

    n_points = len(coords)

    if k >= n_points:
        return np.arange(n_points)

    weights = np.asarray(weights, dtype=float)
    weights = np.maximum(weights, 0)

    if weights.sum() == 0:
        probabilities = None
    else:
        probabilities = weights / weights.sum()

    initial_indices = rng.choice(
        n_points,
        size=k,
        replace=False,
        p=probabilities
    )

    return initial_indices


def update_medoids_by_cluster(coords, weights, labels, k, old_medoid_indices):
    """
    Updates each medoid by choosing an actual point inside the cluster.
    The selected point is the one closest to the weighted centre of that cluster.
    This is a scalable medoid approximation and avoids full O(n^2) distance matrices.
    """
    new_medoid_indices = old_medoid_indices.copy()

    for cluster_id in range(k):
        cluster_indices = np.where(labels == cluster_id)[0]

        if len(cluster_indices) == 0:
            continue

        cluster_coords = coords[cluster_indices]
        cluster_weights = weights[cluster_indices]

        if cluster_weights.sum() <= 0:
            weighted_centre = cluster_coords.mean(axis=0)
        else:
            weighted_centre = np.average(
                cluster_coords,
                axis=0,
                weights=cluster_weights
            )

        distances_to_centre = np.linalg.norm(
            cluster_coords - weighted_centre,
            axis=1
        )

        best_local_index = np.argmin(distances_to_centre)
        new_medoid_indices[cluster_id] = cluster_indices[best_local_index]

    return new_medoid_indices


def scalable_weighted_kmedoids(coords, weights, k, max_iter=8, random_state=42):
    """
    Scalable K-Medoids-style clustering.

    It works like this:
    1. Choose initial medoids from actual points.
    2. Assign every point to its nearest medoid.
    3. Update each medoid to an actual point inside the cluster.
    4. Repeat until stable or max_iter is reached.

    This avoids the very heavy full K-Medoids distance matrix.
    """
    coords = np.asarray(coords, dtype=float)
    weights = np.asarray(weights, dtype=float)

    n_points = len(coords)

    if k >= n_points:
        labels = np.arange(n_points)
        medoid_indices = np.arange(n_points)
        return labels, medoid_indices

    medoid_indices = initialize_weighted_medoids(
        coords=coords,
        weights=weights,
        k=k,
        random_state=random_state
    )

    for iteration in range(max_iter):
        medoid_coords = coords[medoid_indices]

        tree = cKDTree(medoid_coords)
        _, labels = tree.query(coords, k=1)

        new_medoid_indices = update_medoids_by_cluster(
            coords=coords,
            weights=weights,
            labels=labels,
            k=k,
            old_medoid_indices=medoid_indices
        )

        changed = np.sum(new_medoid_indices != medoid_indices)

        print(f"  K-Medoids iteration {iteration + 1}: changed medoids = {changed}")

        medoid_indices = new_medoid_indices

        if changed == 0:
            break

    # Final assignment after last medoid update
    medoid_coords = coords[medoid_indices]
    tree = cKDTree(medoid_coords)
    _, labels = tree.query(coords, k=1)

    return labels, medoid_indices


patrol_allocations = []
assigned_hotspot_points = []

classified_hotspots_df["month"] = pd.to_datetime(classified_hotspots_df["month"])

for month, month_df in classified_hotspots_df.groupby("month"):
    month_df = month_df.copy().reset_index(drop=True)
    month_name = pd.to_datetime(month).strftime("%Y-%m")

    print("\nCreating medoid-based travel-time patrol allocation for:", month_name)
    print("Hotspots in month:", len(month_df))

    # Assign speed based on patrol type from Cell 21
    month_df["speed_kmh"] = month_df["patrol_type"].map({
        "Bike patrol": BIKE_SPEED_KMH,
        "Car patrol": CAR_SPEED_KMH
    })

    if month_df["speed_kmh"].isna().any():
        raise ValueError("Some hotspots have unknown patrol_type and no speed was assigned.")

    # Convert latitude/longitude to approximate x/y kilometres
    ref_lat = month_df[LAT_COL].mean()

    coords_km = latlon_to_xy_km(
        month_df[LAT_COL].values,
        month_df[LON_COL].values,
        ref_lat
    )

    # Convert physical distance into travel-time-scaled coordinates.
    # Since car = 60 km/h and bike = 15 km/h, car is 4 times faster.
    # This means car points are scaled differently than bike points.
    minutes_per_km = month_df["speed_kmh"].apply(speed_to_minutes_per_km).values
    coords_time = coords_km * minutes_per_km[:, None]

    n_hotspots = len(month_df)

    # Number of medoids cannot be larger than number of points
    k = min(CENTROIDS, n_hotspots)
    k = max(1, k)

    print(
        "Combined Bike + Car K-Medoids",
        "| hotspots:", n_hotspots,
        "| requested patrol centres:", CENTROIDS,
        "| used patrol centres:", k
    )

    weights = month_df[DEMAND_COL].values

    labels, medoid_indices = scalable_weighted_kmedoids(
        coords=coords_time,
        weights=weights,
        k=k,
        max_iter=8,
        random_state=42
    )

    month_df["allocation_cluster"] = labels

    assigned_hotspot_points.append(month_df)

    # Create one patrol centre per cluster.
    for cluster_id in sorted(month_df["allocation_cluster"].unique()):
        cluster_points = month_df[month_df["allocation_cluster"] == cluster_id].copy()

        if len(cluster_points) == 0:
            continue

        total_demand = cluster_points[DEMAND_COL].sum()

        medoid_global_index = medoid_indices[int(cluster_id)]
        medoid_row = month_df.iloc[medoid_global_index]

        demand_by_patrol_type = cluster_points.groupby("patrol_type")[DEMAND_COL].sum()
        patrol_type = demand_by_patrol_type.idxmax()

        speed_kmh = BIKE_SPEED_KMH if patrol_type == "Bike patrol" else CAR_SPEED_KMH

        bike_points = (cluster_points["patrol_type"] == "Bike patrol").sum()
        car_points = (cluster_points["patrol_type"] == "Car patrol").sum()

        bike_demand = cluster_points.loc[
            cluster_points["patrol_type"] == "Bike patrol", DEMAND_COL
        ].sum()

        car_demand = cluster_points.loc[
            cluster_points["patrol_type"] == "Car patrol", DEMAND_COL
        ].sum()

        patrol_allocations.append({
            "month": month,
            "patrol_type": patrol_type,
            "speed_kmh": speed_kmh,
            "allocation_cluster": int(cluster_id),

            "centroid_latitude": medoid_row[LAT_COL],
            "centroid_longitude": medoid_row[LON_COL],
            "centroid_lsoa": medoid_row[LSOA_COL],

            "centre_method": "medoid",
            "medoid_original_patrol_type": medoid_row["patrol_type"],
            "medoid_predicted_demand": medoid_row[DEMAND_COL],

            "covered_hotspots": len(cluster_points),
            "bike_hotspots": int(bike_points),
            "car_hotspots": int(car_points),
            "bike_demand": bike_demand,
            "car_demand": car_demand,
            "total_predicted_demand": total_demand,
            "average_predicted_demand": cluster_points[DEMAND_COL].mean(),
            "max_predicted_demand": cluster_points[DEMAND_COL].max()
        })

patrol_allocations_df = pd.DataFrame(patrol_allocations)
assigned_hotspot_points_df = pd.concat(assigned_hotspot_points, ignore_index=True)

patrol_allocations_df["month"] = pd.to_datetime(patrol_allocations_df["month"])
assigned_hotspot_points_df["month"] = pd.to_datetime(assigned_hotspot_points_df["month"])

print("\nMedoid-based travel-time patrol allocation complete.")
print("Total patrol centres:", len(patrol_allocations_df))
print("Assigned hotspot points:", len(assigned_hotspot_points_df))

print("\nPatrol centres by patrol type:")
print(patrol_allocations_df["patrol_type"].value_counts())

patrol_allocations_df.head()


Creating medoid-based travel-time patrol allocation for: 2027-01
Hotspots in month: 11027
Combined Bike + Car K-Medoids | hotspots: 11027 | requested patrol centres: 8156 | used patrol centres: 8156
  K-Medoids iteration 1: changed medoids = 867
  K-Medoids iteration 2: changed medoids = 139
  K-Medoids iteration 3: changed medoids = 5
  K-Medoids iteration 4: changed medoids = 0

Creating medoid-based travel-time patrol allocation for: 2027-02
Hotspots in month: 11037
Combined Bike + Car K-Medoids | hotspots: 11037 | requested patrol centres: 8156 | used patrol centres: 8156
  K-Medoids iteration 1: changed medoids = 865
  K-Medoids iteration 2: changed medoids = 156
  K-Medoids iteration 3: changed medoids = 14
  K-Medoids iteration 4: changed medoids = 0

Creating medoid-based travel-time patrol allocation for: 2027-03
Hotspots in month: 11049
Combined Bike + Car K-Medoids | hotspots: 11049 | requested patrol centres: 8156 | used patrol centres: 8156
  K-Medoids iteration 1: change

,month,patrol_type,speed_kmh,allocation_cluster,centroid_latitude,centroid_longitude,centroid_lsoa,centre_method,medoid_original_patrol_type,medoid_predicted_demand,covered_hotspots,bike_hotspots,car_hotspots,bike_demand,car_demand,total_predicted_demand,average_predicted_demand,max_predicted_demand
0,2027-01-01,Bike patrol,15,0,52.469700,1.745640,E01030258,medoid,Bike patrol,22.35,1,1,0,22.35,0.0,22.35,22.35,22.35
1,2027-01-01,Bike patrol,15,1,52.628748,-1.145442,E01034014,medoid,Bike patrol,21.65,2,2,0,40.50,0.0,40.50,20.25,21.65
2,2027-01-01,Bike patrol,15,2,52.477214,-1.910232,E01033557,medoid,Bike patrol,50.50,1,1,0,50.50,0.0,50.50,50.50,50.50
3,2027-01-01,Bike patrol,15,3,52.528383,-1.375712,E01025835,medoid,Bike patrol,9.18,1,1,0,9.18,0.0,9.18,9.18,9.18
4,2027-01-01,Bike patrol,15,4,51.494434,-0.173900,E01002821,medoid,Bike patrol,29.87,1,1,0,29.87,0.0,29.87,29.87,29.87


In [35]:
import geopandas as gpd
import pandas as pd
import numpy as np

cluster_border_features = []
valid_centroid_markers = []

patrol_allocations_df["month"] = pd.to_datetime(patrol_allocations_df["month"])
assigned_hotspot_points_df["month"] = pd.to_datetime(assigned_hotspot_points_df["month"])

LSOA_BOUNDARIES_FILE = "LSOA_2021_EW_BSC.geojson"

lsoa_gdf = gpd.read_file(LSOA_BOUNDARIES_FILE)

print("Boundary file columns:")
print(lsoa_gdf.columns.tolist())

possible_lsoa_code_cols = [
    "LSOA21CD",
    "LSOA11CD",
    "lsoa21cd",
    "lsoa11cd",
    "LSOA code",
    "LSOA Code",
    "lsoa_code"
]

boundary_lsoa_col = None

for col in possible_lsoa_code_cols:
    if col in lsoa_gdf.columns:
        boundary_lsoa_col = col
        break

if boundary_lsoa_col is None:
    raise ValueError(
        "Could not find the LSOA code column in the boundary file. "
        "Check the printed columns and set boundary_lsoa_col manually."
    )

print("Using boundary LSOA column:", boundary_lsoa_col)
print("Using notebook LSOA column:", LSOA_COL)

lsoa_gdf = lsoa_gdf[[boundary_lsoa_col, "geometry"]].copy()
lsoa_gdf = lsoa_gdf.rename(columns={boundary_lsoa_col: LSOA_COL})

lsoa_gdf[LSOA_COL] = lsoa_gdf[LSOA_COL].astype(str)
assigned_hotspot_points_df[LSOA_COL] = assigned_hotspot_points_df[LSOA_COL].astype(str)
patrol_allocations_df["centroid_lsoa"] = patrol_allocations_df["centroid_lsoa"].astype(str)

lsoa_gdf = lsoa_gdf.to_crs(epsg=27700)

lsoa_gdf["lsoa_marker_point"] = lsoa_gdf.geometry.representative_point()

valid_area_count = 0
valid_marker_count = 0
skipped_invalid_geometry = 0
skipped_missing_lsoa_boundary = 0
skipped_centroid_lsoa_not_in_area = 0
skipped_missing_centroid_lsoa_boundary = 0

allocated_lsoas = set(assigned_hotspot_points_df[LSOA_COL].astype(str).unique())
boundary_lsoas = set(lsoa_gdf[LSOA_COL].astype(str).unique())

missing_lsoas = allocated_lsoas - boundary_lsoas

print("\nAssigned LSOAs:", len(allocated_lsoas))
print("Boundary LSOAs:", len(boundary_lsoas))
print("Assigned LSOAs missing from boundary file:", len(missing_lsoas))
print("Example missing LSOAs:", list(missing_lsoas)[:10])

for month, month_points in assigned_hotspot_points_df.groupby("month"):

    month_name = pd.to_datetime(month).strftime("%Y-%m")
    month_time = pd.to_datetime(month).strftime("%Y-%m-01T00:00:00")

    print("\nCreating strict LSOA patrol areas for:", month_name)

    month_points = month_points.copy()

    month_centroids = patrol_allocations_df[
        patrol_allocations_df["month"] == month
    ].copy()

    if len(month_points) == 0 or len(month_centroids) == 0:
        print("Skipped month because there are no assigned points or centroids.")
        continue

    month_lsoa_assignments = (
        month_points
        .groupby([LSOA_COL, "allocation_cluster"], as_index=False)
        .agg({
            DEMAND_COL: "sum",
            "patrol_type": lambda x: x.value_counts().idxmax()
        })
    )

    month_lsoa_assignments = (
        month_lsoa_assignments
        .sort_values(DEMAND_COL, ascending=False)
        .drop_duplicates(subset=[LSOA_COL], keep="first")
        .copy()
    )

    centroid_meta = month_centroids[[
        "allocation_cluster",
        "patrol_type",
        "speed_kmh",
        "centroid_lsoa",
        "covered_hotspots",
        "bike_hotspots",
        "car_hotspots",
        "bike_demand",
        "car_demand",
        "total_predicted_demand"
    ]].copy()

    centroid_meta = centroid_meta.rename(columns={
        "patrol_type": "centroid_patrol_type"
    })

    month_lsoa_assignments = month_lsoa_assignments.merge(
        centroid_meta,
        on="allocation_cluster",
        how="left"
    )

    month_lsoa_assignments = month_lsoa_assignments.dropna(
        subset=["allocation_cluster", "centroid_patrol_type", "centroid_lsoa"]
    ).copy()

    month_lsoas_gdf = lsoa_gdf[[LSOA_COL, "geometry"]].merge(
        month_lsoa_assignments,
        on=LSOA_COL,
        how="inner"
    )

    print("Assigned LSOAs:", len(month_lsoa_assignments))
    print("Assigned LSOAs with matched boundaries:", len(month_lsoas_gdf))

    skipped_missing_lsoa_boundary += len(month_lsoa_assignments) - len(month_lsoas_gdf)

    if len(month_lsoas_gdf) == 0:
        continue

    dissolved = month_lsoas_gdf.dissolve(
        by="allocation_cluster",
        aggfunc={
            LSOA_COL: "count",
            DEMAND_COL: "sum",
            "centroid_patrol_type": "first",
            "speed_kmh": "first",
            "centroid_lsoa": "first",
            "covered_hotspots": "first",
            "bike_hotspots": "first",
            "car_hotspots": "first",
            "bike_demand": "first",
            "car_demand": "first",
            "total_predicted_demand": "first"
        }
    ).reset_index()

    dissolved = dissolved.rename(columns={
        LSOA_COL: "covered_lsoas",
        DEMAND_COL: "covered_lsoa_demand"
    })

    dissolved["geometry"] = dissolved.geometry.simplify(
        tolerance=40,
        preserve_topology=True
    )

    before_clean = len(dissolved)

    dissolved = dissolved[dissolved.geometry.notna()].copy()
    dissolved["geometry"] = dissolved.geometry.buffer(0)
    dissolved = dissolved[~dissolved.geometry.is_empty].copy()
    dissolved = dissolved[
        dissolved.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ].copy()

    after_clean = len(dissolved)

    print("Clusters before cleaning:", before_clean)
    print("Clusters after cleaning:", after_clean)

    if len(dissolved) == 0:
        continue

    centroid_lsoa_points = lsoa_gdf[
        [LSOA_COL, "lsoa_marker_point"]
    ].copy()

    centroid_lsoa_points = centroid_lsoa_points.rename(columns={
        LSOA_COL: "centroid_lsoa"
    })

    dissolved = dissolved.merge(
        centroid_lsoa_points,
        on="centroid_lsoa",
        how="left"
    )

    dissolved = dissolved[dissolved["lsoa_marker_point"].notna()].copy()

    if len(dissolved) == 0:
        skipped_missing_centroid_lsoa_boundary += after_clean
        print("No centroid LSOA marker points matched official boundaries.")
        continue

    valid_rows = []

    for _, row in dissolved.iterrows():

        geom = row["geometry"]
        centroid_lsoa_point = row["lsoa_marker_point"]

        if geom is None or geom.is_empty:
            skipped_invalid_geometry += 1
            continue

        if geom.geom_type not in ["Polygon", "MultiPolygon"]:
            skipped_invalid_geometry += 1
            continue

        if not (geom.contains(centroid_lsoa_point) or geom.touches(centroid_lsoa_point)):
            skipped_centroid_lsoa_not_in_area += 1
            continue

        valid_rows.append(row)

    if len(valid_rows) == 0:
        print("No clusters kept after centroid LSOA inside-area check.")
        continue

    valid_dissolved = gpd.GeoDataFrame(
        valid_rows,
        geometry="geometry",
        crs="EPSG:27700"
    )

    valid_dissolved_4326 = valid_dissolved.to_crs(epsg=4326)

    marker_points_gdf = gpd.GeoDataFrame(
        valid_dissolved[[
            "allocation_cluster",
            "lsoa_marker_point"
        ]].copy(),
        geometry="lsoa_marker_point",
        crs="EPSG:27700"
    ).to_crs(epsg=4326)

    marker_lookup = marker_points_gdf.set_index("allocation_cluster")["lsoa_marker_point"]

    print("Clusters kept after strict check:", len(valid_dissolved_4326))

    for _, row in valid_dissolved_4326.iterrows():

        cluster_id = row["allocation_cluster"]

        marker_point_4326 = marker_lookup.loc[cluster_id]

        marker_lon = marker_point_4326.x
        marker_lat = marker_point_4326.y

        color = "blue" if row["centroid_patrol_type"] == "Bike patrol" else "red"

        popup = f"""
        <b>Strict LSOA-based patrol area</b><br>
        <b>Month:</b> {month_name}<br>
        <b>Patrol type:</b> {row['centroid_patrol_type']}<br>
        <b>Cluster ID:</b> {row['allocation_cluster']}<br>
        <b>Centroid LSOA:</b> {row['centroid_lsoa']}<br>
        <b>Covered LSOAs:</b> {row['covered_lsoas']}<br>
        <b>Covered LSOA demand:</b> {row['covered_lsoa_demand']:.2f}<br>
        <b>Covered hotspots:</b> {row['covered_hotspots']}<br>
        <b>Bike hotspots:</b> {row['bike_hotspots']}<br>
        <b>Car hotspots:</b> {row['car_hotspots']}<br>
        <b>Bike demand:</b> {row['bike_demand']:.2f}<br>
        <b>Car demand:</b> {row['car_demand']:.2f}<br>
        <b>Total centroid demand:</b> {row['total_predicted_demand']:.2f}
        """

        cluster_border_features.append({
            "type": "Feature",
            "geometry": row["geometry"].__geo_interface__,
            "properties": {
                "times": [month_time],
                "popup": popup,
                "style": {
                    "color": color,
                    "weight": 1.1,
                    "opacity": 0.85,
                    "fillColor": color,
                    "fillOpacity": 0.12
                }
            }
        })

        valid_centroid_markers.append({
            "month": month,
            "allocation_cluster": cluster_id,
            "patrol_type": row["centroid_patrol_type"],
            "centroid_lsoa": row["centroid_lsoa"],
            "latitude": marker_lat,
            "longitude": marker_lon,
            "covered_lsoas": row["covered_lsoas"],
            "covered_lsoa_demand": row["covered_lsoa_demand"],
            "covered_hotspots": row["covered_hotspots"],
            "bike_hotspots": row["bike_hotspots"],
            "car_hotspots": row["car_hotspots"],
            "bike_demand": row["bike_demand"],
            "car_demand": row["car_demand"],
            "total_predicted_demand": row["total_predicted_demand"]
        })

        valid_area_count += 1
        valid_marker_count += 1

valid_centroid_markers_df = pd.DataFrame(valid_centroid_markers)

if len(valid_centroid_markers_df) > 0:
    valid_centroid_markers_df["month"] = pd.to_datetime(valid_centroid_markers_df["month"])

print("\nValid strict LSOA patrol areas created:", valid_area_count)
print("Valid centroid markers created:", valid_marker_count)
print("Skipped invalid geometries:", skipped_invalid_geometry)
print("Skipped missing LSOA boundary matches:", skipped_missing_lsoa_boundary)
print("Skipped missing centroid LSOA boundary:", skipped_missing_centroid_lsoa_boundary)
print("Skipped because centroid LSOA point was outside its area:", skipped_centroid_lsoa_not_in_area)
print("Total area features:", len(cluster_border_features))
print("Total marker rows:", len(valid_centroid_markers_df))

Boundary file columns:
['FID', 'LSOA21CD', 'LSOA21NM', 'LSOA21NMW', 'BNG_E', 'BNG_N', 'LAT', 'LONG', 'GlobalID', 'geometry']
Using boundary LSOA column: LSOA21CD
Using notebook LSOA column: LSOA code

Assigned LSOAs: 11999
Boundary LSOAs: 35672
Assigned LSOAs missing from boundary file: 17
Example missing LSOAs: ['E01013647', 'E01033631', 'E01009881', 'E01013646', 'E01033635', 'E01010371', 'E01032450', 'E01027110', 'E01009635', 'E01009378']

Creating strict LSOA patrol areas for: 2027-01
Assigned LSOAs: 11027
Assigned LSOAs with matched boundaries: 11027
Clusters before cleaning: 8156
Clusters after cleaning: 8156
Clusters kept after strict check: 8152

Creating strict LSOA patrol areas for: 2027-02
Assigned LSOAs: 11037
Assigned LSOAs with matched boundaries: 11037
Clusters before cleaning: 8156
Clusters after cleaning: 8156
Clusters kept after strict check: 8153

Creating strict LSOA patrol areas for: 2027-03
Assigned LSOAs: 11049
Assigned LSOAs with matched boundaries: 11046
Cluster

In [36]:
import folium
import pandas as pd
import json
from branca.element import MacroElement, Template

required_vars = [
    "cluster_border_features",
    "valid_centroid_markers_df"
]

missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    raise RuntimeError(
        f"Missing variables: {missing_vars}. Run Cell 23 first."
    )

if len(cluster_border_features) == 0:
    raise RuntimeError("No valid patrol area features were created.")

if len(valid_centroid_markers_df) == 0:
    raise RuntimeError("No valid centroid markers were created.")

valid_centroid_markers_df["month"] = pd.to_datetime(valid_centroid_markers_df["month"])


def get_feature_month(feature):
    props = feature.get("properties", {})

    if "month_key" in props:
        return props["month_key"]

    if "times" in props and len(props["times"]) > 0:
        return pd.to_datetime(props["times"][0]).strftime("%Y-%m")

    return None


area_features_by_month = {}

for feature in cluster_border_features:
    month_key = get_feature_month(feature)

    if month_key is None:
        continue

    feature["properties"]["month_key"] = month_key

    if month_key not in area_features_by_month:
        area_features_by_month[month_key] = []

    area_features_by_month[month_key].append(feature)


marker_rows_by_month = {}

for month_key, month_df in valid_centroid_markers_df.groupby(
    valid_centroid_markers_df["month"].dt.strftime("%Y-%m")
):
    marker_rows_by_month[month_key] = month_df.copy()


# All available months
all_months = sorted(
    set(area_features_by_month.keys()) |
    set(marker_rows_by_month.keys())
)

print("Months in map:")
print(all_months)

if len(all_months) == 0:
    raise RuntimeError("No months found in area or marker features.")


# Base map
map_center = [
    valid_centroid_markers_df["latitude"].mean(),
    valid_centroid_markers_df["longitude"].mean()
]

patrol_map = folium.Map(
    location=map_center,
    zoom_start=9,
    tiles="cartodbpositron"
)


# Style function for LSOA patrol areas
def area_style(feature):
    return feature.get("properties", {}).get("style", {
        "color": "black",
        "weight": 1.0,
        "opacity": 0.8,
        "fillColor": "black",
        "fillOpacity": 0.10
    })


# Create one FeatureGroup layer per month
month_layer_names = {}

for month_key in all_months:

    month_layer = folium.FeatureGroup(
        name=f"Patrol allocation {month_key}",
        show=True
    )

    # Areas for this month
    month_area_features = area_features_by_month.get(month_key, [])

    if len(month_area_features) > 0:
        folium.GeoJson(
            {
                "type": "FeatureCollection",
                "features": month_area_features
            },
            name=f"Areas {month_key}",
            style_function=area_style,
            tooltip=folium.GeoJsonTooltip(
                fields=["month_key"],
                aliases=["Month:"],
                labels=True
            )
        ).add_to(month_layer)

    month_markers = marker_rows_by_month.get(month_key, pd.DataFrame())

    for _, row in month_markers.iterrows():

        color = "blue" if row["patrol_type"] == "Bike patrol" else "red"

        popup_text = f"""
        <b>Patrol allocation point</b><br>
        <b>Patrol type:</b> {row['patrol_type']}<br>
        <b>Month:</b> {month_key}<br>
        <b>Cluster ID:</b> {row['allocation_cluster']}<br>
        <b>Covered LSOAs:</b> {row['covered_lsoas']}<br>
        <b>Covered LSOA demand:</b> {row['covered_lsoa_demand']:.2f}<br>
        <b>Covered hotspots:</b> {row['covered_hotspots']}<br>
        <b>Bike hotspots:</b> {row['bike_hotspots']}<br>
        <b>Car hotspots:</b> {row['car_hotspots']}<br>
        <b>Bike demand:</b> {row['bike_demand']:.2f}<br>
        <b>Car demand:</b> {row['car_demand']:.2f}<br>
        <b>Total predicted demand:</b> {row['total_predicted_demand']:.2f}
        """

        folium.CircleMarker(
            location=[
                float(row["latitude"]),
                float(row["longitude"])
            ],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.95,
            opacity=0.95,
            popup=folium.Popup(popup_text, max_width=350),
            tooltip=f"{row['patrol_type']} | {int(row['covered_lsoas'])} LSOAs"
        ).add_to(month_layer)

    month_layer.add_to(patrol_map)

    month_layer_names[month_key] = month_layer.get_name()

    print(
        month_key,
        "| area features:",
        len(month_area_features),
        "| markers:",
        len(month_markers)
    )


months_js = json.dumps(all_months)

layer_js_entries = []

for month_key, layer_var in month_layer_names.items():
    layer_js_entries.append(f'"{month_key}": {layer_var}')

layers_js = "{\n" + ",\n".join(layer_js_entries) + "\n}"

map_var = patrol_map.get_name()

controller_template = f"""
{{% macro html(this, kwargs) %}}
<div id="patrolMonthControl" style="
position: fixed;
bottom: 50px;
left: 50px;
width: 390px;
height: 175px;
background-color: white;
z-index: 9999;
font-size: 14px;
border: 2px solid grey;
padding: 10px;
box-shadow: 2px 2px 5px rgba(0,0,0,0.35);
">
<b>Monthly Patrol Allocation</b><br>
<span style="color:blue;">●</span> Bike allocation point<br>
<span style="color:red;">●</span> Car allocation point<br>
<span style="color:blue;">▭</span> Bike LSOA patrol area<br>
<span style="color:red;">▭</span> Car LSOA patrol area<br>
<br>

<button id="patrolPrevMonth" type="button">◀</button>
<button id="patrolPlayPause" type="button">▶ Play</button>
<button id="patrolNextMonth" type="button">▶</button>

&nbsp;&nbsp;
<b><span id="patrolSelectedMonth">{all_months[0]}</span></b>
<br>

<input
    id="patrolMonthSlider"
    type="range"
    min="0"
    max="{len(all_months) - 1}"
    value="0"
    step="1"
    style="width: 350px; margin-top: 8px;"
>
</div>
{{% endmacro %}}

{{% macro script(this, kwargs) %}}

var patrolMonths = {months_js};
var patrolMonthLayers = {layers_js};
var patrolMap = {map_var};

var patrolCurrentIndex = 0;
var patrolIsPlaying = false;
var patrolTimer = null;
var patrolDelayMs = 1200;

function patrolHideAllMonths() {{
    for (var i = 0; i < patrolMonths.length; i++) {{
        var month = patrolMonths[i];
        var layer = patrolMonthLayers[month];

        if (patrolMap.hasLayer(layer)) {{
            patrolMap.removeLayer(layer);
        }}
    }}
}}

function patrolShowMonth(index) {{
    if (index < 0) {{
        index = patrolMonths.length - 1;
    }}

    if (index >= patrolMonths.length) {{
        index = 0;
    }}

    patrolCurrentIndex = index;

    var selectedMonth = patrolMonths[patrolCurrentIndex];

    patrolHideAllMonths();

    patrolMap.addLayer(patrolMonthLayers[selectedMonth]);

    var label = document.getElementById("patrolSelectedMonth");
    if (label) {{
        label.innerHTML = selectedMonth;
    }}

    var slider = document.getElementById("patrolMonthSlider");
    if (slider) {{
        slider.value = patrolCurrentIndex;
    }}
}}

function patrolNextMonth() {{
    patrolShowMonth(patrolCurrentIndex + 1);
}}

function patrolPrevMonth() {{
    patrolShowMonth(patrolCurrentIndex - 1);
}}

function patrolStartPlaying() {{
    patrolIsPlaying = true;

    var playButton = document.getElementById("patrolPlayPause");
    if (playButton) {{
        playButton.innerHTML = "⏸ Pause";
    }}

    patrolTimer = setInterval(function() {{
        patrolNextMonth();
    }}, patrolDelayMs);
}}

function patrolStopPlaying() {{
    patrolIsPlaying = false;

    var playButton = document.getElementById("patrolPlayPause");
    if (playButton) {{
        playButton.innerHTML = "▶ Play";
    }}

    if (patrolTimer !== null) {{
        clearInterval(patrolTimer);
        patrolTimer = null;
    }}
}}

function patrolTogglePlay() {{
    if (patrolIsPlaying) {{
        patrolStopPlaying();
    }} else {{
        patrolStartPlaying();
    }}
}}

// Wait until the page has loaded, then connect controls
setTimeout(function() {{

    var controlBox = document.getElementById("patrolMonthControl");

    if (controlBox) {{
        L.DomEvent.disableClickPropagation(controlBox);
        L.DomEvent.disableScrollPropagation(controlBox);
    }}

    var slider = document.getElementById("patrolMonthSlider");
    var prevButton = document.getElementById("patrolPrevMonth");
    var nextButton = document.getElementById("patrolNextMonth");
    var playButton = document.getElementById("patrolPlayPause");

    if (slider) {{
        slider.addEventListener("input", function(e) {{
            patrolShowMonth(parseInt(e.target.value));
        }});
    }}

    if (prevButton) {{
        prevButton.addEventListener("click", function() {{
            patrolPrevMonth();
        }});
    }}

    if (nextButton) {{
        nextButton.addEventListener("click", function() {{
            patrolNextMonth();
        }});
    }}

    if (playButton) {{
        playButton.addEventListener("click", function() {{
            patrolTogglePlay();
        }});
    }}

    // Show only the first month when the map opens
    patrolShowMonth(0);

}}, 500);

{{% endmacro %}}
"""

controller = MacroElement()
controller._template = Template(controller_template)
patrol_map.get_root().add_child(controller)

print("Months:", all_months)

Months in map:
['2027-01', '2027-02', '2027-03', '2027-04', '2027-05', '2027-06', '2027-07', '2027-08', '2027-09', '2027-10', '2027-11', '2027-12']
2027-01 | area features: 8152 | markers: 8152
2027-02 | area features: 8153 | markers: 8153
2027-03 | area features: 8152 | markers: 8152
2027-04 | area features: 8150 | markers: 8150
2027-05 | area features: 8152 | markers: 8152
2027-06 | area features: 8148 | markers: 8148
2027-07 | area features: 8145 | markers: 8145
2027-08 | area features: 8145 | markers: 8145
2027-09 | area features: 8144 | markers: 8144
2027-10 | area features: 8149 | markers: 8149
2027-11 | area features: 8146 | markers: 8146
2027-12 | area features: 8146 | markers: 8146
Months: ['2027-01', '2027-02', '2027-03', '2027-04', '2027-05', '2027-06', '2027-07', '2027-08', '2027-09', '2027-10', '2027-11', '2027-12']


In [37]:
patrol_map.save(OUTPUT_ALLOCATION_MAP_HTML)

print("Map saved as:")
print(OUTPUT_ALLOCATION_MAP_HTML)

Map saved as:
police_officers_allocation.html
